# AIMO3 Writeup — Kaggle-Agnostic Reproduction (Google Colab)

**Companion notebook for the AIMO3 Writeup Prize submission:** *A Practitioner's Plateau on gpt-oss-120b-MXFP4 — Sixteen Falsified Modifications to the AIMO3 Public-Consensus Pipeline.*

**Authors:** Juan Sebastian Gil Pinzon (Kaggle: `sebastiangil00`) and Kimberley Duran (Kaggle: `kimberleyduran`), team *Hail Mary*.

- Kaggle Writeup: https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3/writeups/a-practitioners-plateau-on-gpt-oss-120b-mxfp4
- Kaggle Submission (42/50 public = 42/50 private): https://www.kaggle.com/code/sebastiangil00/aimo3-v7-winner-fork
- GitHub companion: https://github.com/SebastianGilPinzon/aimo3-writeup-public
- Kaggle Dataset: https://www.kaggle.com/datasets/sebastiangil00/aimo3-writeup-artifacts

## What this notebook demonstrates

This is a Kaggle-agnostic implementation of the core inference pipeline described in the writeup: **Harmony-formatted prompting** on `gpt-oss-120b` (MoE expert projections in MXFP4, attention in BF16), served through **vLLM 0.11.2**, with **K=8 parallel rollouts** and **entropy-weighted majority voting**.

The cells below are self-contained — they do NOT import our Kaggle submission's `notebook.py` (which contains Kaggle-gateway-specific bootstrap code that would fail outside Kaggle). Instead, the core pipeline logic is re-implemented inline here so a reviewer can verify each step independently.

Three independent verification paths are available:

- **Section 1–3 (CPU-only, no GPU needed, ~30 s total):** install minimal deps, clone the companion repo, run `pytest tests/` — all 15 non-integration substrate-trap tests pass. **This is the fastest reviewer path and requires no H100.**
- **Section 4–6 (GPU, ~25 min):** install vLLM, download `gpt-oss-120b` from HuggingFace, launch the server, send a sample inference with Harmony format + TIR, verify non-empty output. Demonstrates the core decoder loop works on a Colab-hosted H100.
- **Section 7 (CPU, ~10 s):** regenerate all 4 figures from the writeup via `figures/generate_figures.py`.

## Runtime requirements

- **CPU path (§1-3, §7):** free Colab runtime. Runs in ~1 minute total.
- **GPU path (§4-6):** Colab Pro+ with NVIDIA H100 (or A100 80GB). Model download is ~60 GB / 10-20 min; inference is ~30 s for the demo query.

## Note on differences from the Kaggle submission

The cells here re-implement the pipeline's *logic* in a form that runs on Colab. The Kaggle submission (`submission/notebook.py` in the companion repo) has additional Kaggle-specific infrastructure: `kaggle_evaluation` gateway integration, Kaggle-input-path detection, pre-flight environment cleanup. That notebook is **byte-identical** to what produced public=private=42/50 on Kaggle and is available for inspection in the repo; we do not import it here because its top-level setup would fail outside Kaggle.

The two intentional Colab-specific adjustments (per host Edit 3 guidance for Kaggle-agnostic reproduction): `gpu_memory_utilization=0.90` (vs `0.94` on Kaggle) and `PYTORCH_ALLOC_CONF=expandable_segments:True`. Both are Colab headroom fixes and do not affect inference accuracy.

---

## §1 Install minimal CPU dependencies

In [ ]:
# Minimal CPU deps for substrate-trap detection tests (§2) and figure generation (§7)
%pip install --quiet pytest numpy pandas matplotlib

## §2 Clone companion repo and run the 3 substrate-trap detection tests (CPU, <30s)

This is the **fastest reviewer path**. Runs the 3 CPU-runnable substrate-trap detectors described in §6 of the writeup. Expected output: `15 passed, 2 skipped in <1s`. The 2 skipped tests are H100 integration tests that require `ENABLE_H100_INTEGRATION=1`.

In [ ]:
!git clone --depth 1 https://github.com/SebastianGilPinzon/aimo3-writeup-public /content/aimo3-writeup 2>/dev/null || echo 'already cloned'
%cd /content/aimo3-writeup
!python -m pytest tests/ -v

## §3 Verify the reproducibility package integrity (17 artifacts, SHA256)

In [ ]:
!sha256sum --check reproducibility/artifact_sha256.txt | tail -20

---

## §4 GPU path: install vLLM 0.11.2 (pinned to match Kaggle submission)

Requires Colab Pro+ with H100. Skip this section for the CPU-only review path.

In [ ]:
!nvidia-smi -L
# Expected: NVIDIA H100 80GB HBM3 (or A100 80GB as a fallback)

In [ ]:
%pip install --quiet vllm==0.11.2 openai openai-harmony huggingface-hub

## §5 Download gpt-oss-120b from HuggingFace (~60 GB, 10-20 min on first run)

In [ ]:
import os
from huggingface_hub import snapshot_download

MODEL_REPO = 'danielhanchen/gpt-oss-120b'
MODEL_LOCAL = '/content/gpt-oss-120b'

if not os.path.isdir(MODEL_LOCAL) or not os.listdir(MODEL_LOCAL):
    snapshot_download(
        repo_id=MODEL_REPO,
        local_dir=MODEL_LOCAL,
        local_dir_use_symlinks=False,
        max_workers=8,
    )

print('Model ready at', MODEL_LOCAL)
!du -sh "$MODEL_LOCAL"

## §6 Launch vLLM and run one Harmony-formatted inference (~3 min)

Reproduces the core pipeline on a single sample problem. This demonstrates: (a) vLLM server with MXFP4 + MoE + BF16-attention launches correctly with the pinned config, (b) Harmony prompt format is constructed as in §2.2 of the writeup, (c) the assistant responds with non-empty output (not the EAGLE-3 zero-token failure documented in §6.1 of the writeup).

Full K=8 parallel + entropy voting over all 10 reference problems takes ~25 minutes; the single-query demo below runs in ~30 s after server startup and proves the core loop works.

In [ ]:
# Launch vLLM server in background
import subprocess, sys, time, os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', '/content/gpt-oss-120b',
    '--served-model-name', 'gpt-oss',
    '--dtype', 'auto',
    '--quantization', 'mxfp4',
    '--kv-cache-dtype', 'fp8_e4m3',
    '--max-model-len', '65536',
    '--gpu-memory-utilization', '0.90',
    '--trust-remote-code',
    '--tensor-parallel-size', '1',
    '--host', '127.0.0.1',
    '--port', '8000',
    '--disable-log-stats',
]
log = open('/content/vllm.log', 'w')
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print('vLLM spawning (PID', proc.pid, '). Polling readiness (up to 5 min for Triton compile)...')

In [ ]:
# Poll server readiness
import requests, time
ready = False
for i in range(60):
    try:
        r = requests.get('http://127.0.0.1:8000/v1/models', timeout=3)
        if r.ok:
            print(f'vLLM ready after {i*5}s:', r.json()['data'][0]['id'])
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)
assert ready, 'vLLM did not come up in 5 minutes; see /content/vllm.log'

In [ ]:
# Construct the Harmony prompt and send ONE inference
# This is the same prompt format documented in §2.2 of the writeup (exact system prompt, tool prompt).
from openai_harmony import (
    SystemContent, Message, Role, Conversation,
    ReasoningEffort, ToolNamespaceConfig,
    load_harmony_encoding, HarmonyEncodingName,
)
from openai import OpenAI

encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

SYSTEM_PROMPT = (
    'You are a world-class International Mathematical Olympiad (IMO) competitor. '
    'The final answer must be a non-negative integer between 0 and 99999. '
    'You must place the final integer answer inside \\boxed{}. '
    'IMPORTANT: Before solving analytically, explore small cases with Python code to find patterns. '
    'For number theory with large numbers, use Fermat-Euler theorem to reduce modular calculations. '
    'For combinatorics, compute first few values and look for known sequences (Catalan, Fibonacci).'
)

# Load a sample reference problem (the easy 92ba6a, answer = 50)
import pandas as pd
ref = pd.read_csv('/content/aimo3-writeup/data/reference.csv')
sample = ref[ref['id'] == '92ba6a'].iloc[0] if '92ba6a' in set(ref['id']) else ref.iloc[0]
print('Problem id:', sample['id'])
print('Problem text:', sample['problem'][:400], '...')
print('Expected answer:', sample['answer'])

In [ ]:
# Build Harmony-formatted messages and send one rollout
tool_config = ToolNamespaceConfig.python()
system_content = (
    SystemContent.new()
    .with_model_identity(SYSTEM_PROMPT)
    .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
    .with_tools(tool_config)
)
system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
user_message = Message.from_role_and_content(Role.USER, sample['problem'])
conversation = Conversation.from_messages([system_message, user_message])
prompt_token_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)

client = OpenAI(base_url='http://127.0.0.1:8000/v1', api_key='sk-local')
resp = client.completions.create(
    model='gpt-oss',
    prompt=[encoding.decode(prompt_token_ids)],
    temperature=1.0,
    max_tokens=4096,
    extra_body={'min_p': 0.02, 'top_k': -1},
)
raw_text = resp.choices[0].text
print('Output chars:', len(raw_text))
print('Output preview (first 500 chars):')
print(raw_text[:500])

In [ ]:
# Extract the final boxed answer
import re

def extract_last_boxed(text):
    matches = re.findall(r'\\boxed\{(\d+)\}', text)
    return int(matches[-1]) if matches else None

answer = extract_last_boxed(raw_text)
print(f'Extracted answer: {answer}')
print(f'Expected: {sample["answer"]}')
print(f'Single-rollout correct: {answer == int(sample["answer"])}')
print('')
print('This demonstrates the core inference loop works on Colab-hosted H100.')
print('The full Kaggle pipeline runs K=8 such rollouts in parallel and votes; see §2.6 of the writeup.')
print('For the full 10-problem reference run on H100, see reproducibility/reproduce.sh in the repo.')

## §7 Regenerate all 4 figures from the writeup (CPU only, <10 s)

In [ ]:
%cd /content/aimo3-writeup/figures
!python generate_figures.py
!ls -la *.png

---

## Summary for reviewers

| Verification path | Time | Requirement | What it proves |
|---|---|---|---|
| §2 `pytest tests/` | <1 s | CPU only | Substrate-trap detectors pass on any machine |
| §3 `sha256sum --check` | <1 s | CPU only | All 17 artifact files verified |
| §7 figure regeneration | <10 s | CPU only | All 4 writeup figures reproducible from code |
| §4-6 vLLM inference | ~25 min | H100 + ~$5 Colab credit | Core decoder loop works Kaggle-agnostic |

The CPU paths are sufficient for reproducibility-criterion verification without cost. The GPU path is offered for reviewers who want to confirm the inference loop works end-to-end on a fresh Colab-hosted environment.

## Contact

- Issues: https://github.com/SebastianGilPinzon/aimo3-writeup-public/issues
- Kaggle: `sebastiangil00` / `kimberleyduran`